# save-results

Saves standardised result tables for one experimental condition (NW topology + params).
Run this notebook once after each batch of simulations completes.

Output directory: `results/summary/{topology}/{net_params}/{exp_id}/`

| File | Contents | Granularity |
|------|----------|-------------|
| `selected_seeds.json` | The 50 seeds used (reproducible) | — |
| `nw_stats_5000step.csv` | gamma, clustering, density, **modularity** per seed × direction × step | 5000-step snapshots |
| `post_count_100step.csv` | Mean posts per direction-relative class per seed × direction | 100-step windows |
| `post_prob_100step.csv` | Mean postProb per direction-relative class per seed × direction | 100-step windows |
| `follower_composition_5000step.csv` | Target hub follower opinion class fractions per seed × direction × step | 5000-step snapshots |
| `unfollow_by_class.csv` | Unfollow counts pre/post manipulation per seed × direction | 2 windows |
| `posting_prob_increase_by_class.csv` | Agents with ↑ postProb, count + rate per seed × direction | 2 windows |
| `target_hub_metrics_5000step.csv` | Structural & follower-opinion metrics of the target hub per seed × direction × step | 5000-step snapshots |
| `connectivity_5000step.csv` | `lambda2_G`, `lambda2_G_minus_target`, `delta_lambda2`, `target_is_ap` per seed × direction × step | 5000-step snapshots |

**Class labels** are always direction-relative (`far_opposite` … `far_target`), never raw bin indices.

In [1]:
import os
import re
import glob
import json
import yaml
import hashlib
import numpy as np
import pandas as pd
import networkx as nx
import shutil
import xml.etree.ElementTree as ET

RESULTS_DIR  = "./results"
SUMMARY_ROOT = "./results/summary"
N_AGENTS     = 1000
MAX_SEEDS    = 50
RNG_SEED     = 42   # fixed for reproducibility

GEXF_STEPS   = [0, 5000, 10000, 15000, 20000, 25000, 30000, 35000, 40000]
BINS         = ["bin_0", "bin_1", "bin_2", "bin_3", "bin_4"]
GROUP_ORDER  = ['far_opposite', 'near_opposite', 'neutral', 'near_target', 'far_target']

_POS_MAP = {
     1.0: {0: 'far_opposite', 1: 'near_opposite', 2: 'neutral', 3: 'near_target', 4: 'far_target'},
    -1.0: {0: 'far_target',   1: 'near_target',   2: 'neutral', 3: 'near_opposite', 4: 'far_opposite'},
}

# ---------------------------------------------------------------------------
# GEXF helpers
# ---------------------------------------------------------------------------
def _strip_ns(root):
    for elem in root.iter():
        if '}' in elem.tag:
            elem.tag = elem.tag.split('}', 1)[1]

def _gexf_path(run_dir, step):
    fs = glob.glob(os.path.join(run_dir, 'GEXF', '*', f'step_{step}.gexf'))
    return fs[0] if fs else None

def _gexf_has_target(fpath):
    try:
        tree = ET.parse(fpath); root = tree.getroot(); _strip_ns(root)
        attr_map = {a.get('id'): a.get('title')
                    for a in root.findall(".//attributes[@class='node']/attribute")}
        for node in root.findall('.//node'):
            for av in node.findall('.//attvalue'):
                if attr_map.get(av.get('for'), av.get('for')) == 'target' \
                        and av.get('value', '').lower() == 'true':
                    return True
    except Exception:
        pass
    return False

def parse_gexf_full(fpath):
    """Returns (DiGraph, target_id_or_None, {nid: {opinionclass, postprob}})."""
    try:
        tree = ET.parse(fpath); root = tree.getroot(); _strip_ns(root)
        attr_map = {a.get('id'): a.get('title')
                    for a in root.findall(".//attributes[@class='node']/attribute")}
        G = nx.DiGraph()
        target_id = None
        agents = {}
        for node in root.findall('.//node'):
            nid = node.get('id')
            G.add_node(nid)
            d = {'opinionclass': -1, 'postprob': np.nan}
            for av in node.findall('.//attvalue'):
                t, v = attr_map.get(av.get('for'), av.get('for')), av.get('value', '')
                if t == 'opinionClass':
                    try: d['opinionclass'] = int(v)
                    except: pass
                elif t == 'postProb':
                    try: d['postprob'] = float(v)
                    except: pass
                elif t == 'target' and v.lower() == 'true':
                    target_id = nid
            agents[nid] = d
        for edge in root.findall('.//edge'):
            G.add_edge(edge.get('source'), edge.get('target'))
        return G, target_id, agents
    except Exception:
        return None, None, {}

# ---------------------------------------------------------------------------
# MLE power-law gamma on in-degree (Clauset et al. 2009)
# ---------------------------------------------------------------------------
def _mle_powerlaw(k):
    k = k[k >= 1]
    if len(k) < 5:
        return np.nan
    candidates = np.unique(k)
    best_ks, best_kmin = np.inf, candidates[0]
    for kmin in candidates:
        tail = k[k >= kmin]
        n = len(tail)
        if n < 5: break
        g = 1.0 + n / np.sum(np.log(tail / (kmin - 0.5)))
        xs = np.sort(tail)
        ecdf = np.arange(1, n + 1) / n
        tcdf = 1.0 - (xs / kmin) ** (-(g - 1))
        ks = np.max(np.abs(ecdf - tcdf))
        if ks < best_ks:
            best_ks, best_kmin = ks, kmin
    tail_f = k[k >= best_kmin]
    n_f = len(tail_f)
    if n_f < 5: return np.nan
    return 1.0 + n_f / np.sum(np.log(tail_f / (best_kmin - 0.5)))

# ---------------------------------------------------------------------------
# Config loader (mirrors result-summary.ipynb)
# ---------------------------------------------------------------------------
def load_yaml_manual(fpath):
    config = {}; current_section = None
    try:
        with open(fpath, 'r') as f:
            for line in f:
                raw = line.rstrip(); line = raw.strip()
                if not line or line.startswith('#'): continue
                indent = len(raw) - len(line)
                if ':' in line:
                    key, val = line.split(':', 1)
                    key = key.strip(); val = val.strip().split('#')[0].strip()
                    if val.isdigit(): val = int(val)
                    elif val.replace('.','',1).isdigit(): val = float(val)
                    elif val.lower() == 'true': val = True
                    elif val.lower() == 'false': val = False
                    elif val.startswith('"') and val.endswith('"'): val = val[1:-1]
                    if indent == 0:
                        if not val: current_section = key; config[current_section] = {}
                        else: config[key] = val; current_section = None
                    elif current_section:
                        config[current_section][key] = val
    except: pass
    return config

print('Helpers loaded.')


Helpers loaded.


In [2]:
# ---------------------------------------------------------------------------
# 1. Detect valid seeds (target node found at step 20000)
# ---------------------------------------------------------------------------
def build_valid_seeds(results_dir, check_step=20000):
    seed_dirs = glob.glob(os.path.join(results_dir, 'run_*_dir_*'))
    seeds_found = set()
    for d in seed_dirs:
        m = re.match(r'run_(\d+)_dir_', os.path.basename(d))
        if m: seeds_found.add(int(m.group(1)))

    valid, excluded = [], []
    for seed in sorted(seeds_found):
        gexf_files = glob.glob(
            os.path.join(results_dir, f'run_{seed}_dir_*', 'GEXF', '*', f'step_{check_step}.gexf'))
        if any(_gexf_has_target(f) for f in gexf_files):
            valid.append(seed)
        else:
            excluded.append(seed)
    print(f'Valid seeds  : {len(valid)}  {valid}')
    print(f'Excluded     : {len(excluded)}  {excluded}')
    return valid

ALL_VALID_SEEDS = build_valid_seeds(RESULTS_DIR)

# ---------------------------------------------------------------------------
# 2. Select up to MAX_SEEDS with fixed RNG
# ---------------------------------------------------------------------------
rng = np.random.default_rng(RNG_SEED)
if len(ALL_VALID_SEEDS) <= MAX_SEEDS:
    SELECTED_SEEDS = sorted(ALL_VALID_SEEDS)
    print(f'All {len(SELECTED_SEEDS)} valid seeds used (< {MAX_SEEDS}).')
else:
    SELECTED_SEEDS = sorted(rng.choice(ALL_VALID_SEEDS, MAX_SEEDS, replace=False).tolist())
    print(f'Randomly selected {MAX_SEEDS} seeds from {len(ALL_VALID_SEEDS)}: {SELECTED_SEEDS}')

# ---------------------------------------------------------------------------
# 3. Determine output directory from config.yaml
# ---------------------------------------------------------------------------
cfg_path = './config.yaml'
config   = load_yaml_manual(cfg_path) if os.path.exists(cfg_path) else {}

param_str = json.dumps(config, sort_keys=True, default=str)
exp_id    = hashlib.md5(param_str.encode()).hexdigest()[:8]

topo    = config.get('topology', 'Unknown')
net_p   = config.get('network_params', {})
net_str = '_'.join(f'{k}_{net_p[k]}' for k in sorted(net_p)) if net_p else 'default'

DEST_DIR = os.path.join(SUMMARY_ROOT, topo, net_str, exp_id)
os.makedirs(DEST_DIR, exist_ok=True)
print(f'Output dir: {DEST_DIR}')

# Save selected seeds
shutil.copy(cfg_path, os.path.join(DEST_DIR, 'config.yaml'))
print('config.yaml saved.')

with open(os.path.join(DEST_DIR, 'selected_seeds.json'), 'w') as f:
    json.dump({'rng_seed': RNG_SEED, 'n_valid': len(ALL_VALID_SEEDS),
               'selected': SELECTED_SEEDS}, f, indent=2)
print('selected_seeds.json saved.')


Valid seeds  : 25  [1, 3, 4, 6, 12, 17, 23, 30, 32, 35, 36, 39, 41, 43, 51, 54, 57, 65, 67, 69, 77, 78, 79, 85, 96]
Excluded     : 75  [0, 2, 5, 7, 8, 9, 10, 11, 13, 14, 15, 16, 18, 19, 20, 21, 22, 24, 25, 26, 27, 28, 29, 31, 33, 34, 37, 38, 40, 42, 44, 45, 46, 47, 48, 49, 50, 52, 53, 55, 56, 58, 59, 60, 61, 62, 63, 64, 66, 68, 70, 71, 72, 73, 74, 75, 76, 80, 81, 82, 83, 84, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 97, 98, 99]
All 25 valid seeds used (< 50).
Output dir: ./results/summary/ER/p_0.003/dad3c91b
config.yaml saved.
selected_seeds.json saved.


In [3]:
# ---------------------------------------------------------------------------
# Item 1: NW stats at 5000-step intervals
# Source: degrees/degree_result_{step}.csv  -> gamma, mean_in_degree, density
#         clusterings/clustering_result_{step}.csv -> avg_clustering
#         GEXF (step snapshot, target included)   -> modularity
# ---------------------------------------------------------------------------

def _compute_modularity(G):
    """
    Directed modularity Q (Leicht & Newman 2008) with Louvain community detection.

    Communities are detected on the undirected projection (Louvain / greedy
    fallback), then Q is evaluated on the original directed graph using the
    directed null model:

        Q = (1/m) * sum_ij [A_ij - k_i^out * k_j^in / m] * delta(c_i, c_j)

    Target node is included (consistent with degree/clustering CSVs from Java).
    Returns nan if graph is too small or detection fails.
    """
    if G is None or G.number_of_nodes() < 4:
        return np.nan
    try:
        G_und = G.to_undirected()
        try:
            communities = nx.community.louvain_communities(G_und, seed=42)
        except AttributeError:
            communities = list(nx.community.greedy_modularity_communities(G_und))
        return nx.community.modularity(G, communities)
    except Exception:
        return np.nan


nw_records = []

for seed in SELECTED_SEEDS:
    for target_sign in [1.0, -1.0]:
        run_dir = os.path.join(RESULTS_DIR, f'run_{seed}_dir_{target_sign}')
        if not os.path.isdir(run_dir):
            continue

        for step in GEXF_STEPS:
            rec = {'seed': seed, 'target_sign': target_sign, 'step': step,
                   'gamma': np.nan, 'mean_in_degree': np.nan,
                   'density': np.nan, 'avg_clustering': np.nan,
                   'modularity': np.nan}

            # degree CSV
            deg_path = os.path.join(run_dir, 'degrees', f'degree_result_{step}.csv')
            if os.path.exists(deg_path):
                try:
                    df_deg = pd.read_csv(deg_path)
                    in_deg = df_deg['inDegree'].values.astype(float)
                    rec['gamma']          = _mle_powerlaw(in_deg)
                    rec['mean_in_degree'] = float(np.mean(in_deg))
                    rec['density']        = float(np.mean(in_deg) / (N_AGENTS - 1))
                except Exception as e:
                    print(f'[seed={seed} dir={target_sign} step={step}] degree error: {e}')

            # clustering CSV (not saved at step 0 by default)
            clu_path = os.path.join(run_dir, 'clusterings', f'clustering_result_{step}.csv')
            if os.path.exists(clu_path):
                try:
                    df_clu = pd.read_csv(clu_path)
                    rec['avg_clustering'] = float(df_clu['clusteringCoefficient'].mean())
                except Exception as e:
                    print(f'[seed={seed} dir={target_sign} step={step}] clustering error: {e}')

            # GEXF -> modularity (target included)
            fp = _gexf_path(run_dir, step)
            if fp:
                try:
                    G, _, _ = parse_gexf_full(fp)
                    rec['modularity'] = _compute_modularity(G)
                except Exception as e:
                    print(f'[seed={seed} dir={target_sign} step={step}] modularity error: {e}')

            nw_records.append(rec)

df_nw = pd.DataFrame(nw_records)
out_path = os.path.join(DEST_DIR, 'nw_stats_5000step.csv')
df_nw.to_csv(out_path, index=False)
print(f'nw_stats_5000step.csv saved  ({len(df_nw)} rows)')
df_nw.head()

nw_stats_5000step.csv saved  (450 rows)


,seed,target_sign,step,gamma,mean_in_degree,density,avg_clustering,modularity
0,1,1.0,0,2.280291,2.901,0.002904,NaN,0.402999
1,1,1.0,5000,3.999887,6.880,0.006887,0.200021,0.563189
2,1,1.0,10000,2.654957,6.895,0.006902,0.271064,0.653912
3,1,1.0,15000,2.797278,6.933,0.006940,0.311382,0.697541
4,1,1.0,20000,2.793506,6.937,0.006944,0.323833,0.705622


In [4]:
# ---------------------------------------------------------------------------
# Item 2: Post count per direction-relative class (100-step windows)
# Source: posts/post_result_*.csv
# ---------------------------------------------------------------------------
WINDOW = 100

def load_posts_timeseries(run_dir, target_sign, window=WINDOW):
    """Returns DataFrame with columns [seed, target_sign, step, far_opposite, ..., far_target]."""
    post_dir = os.path.join(run_dir, 'posts')
    files = glob.glob(os.path.join(post_dir, 'post_result_*.csv'))
    if not files: return None
    dfs = []
    for f in files:
        try: dfs.append(pd.read_csv(f))
        except: pass
    if not dfs: return None
    df = pd.concat(dfs).sort_values('step').reset_index(drop=True)

    pmap = _POS_MAP[target_sign]
    # rename bin_i → relative group
    bin_to_group = {f'bin_{i}': pmap[i] for i in range(5)}
    for b, g in bin_to_group.items():
        if b in df.columns:
            df[g] = df.get(g, 0) + df[b]

    df['window_idx'] = df['step'] // window
    agg = df.groupby('window_idx')[GROUP_ORDER].mean()
    agg['step'] = agg.index * window
    return agg.reset_index(drop=True)

post_count_records = []
for seed in SELECTED_SEEDS:
    for target_sign in [1.0, -1.0]:
        run_dir = os.path.join(RESULTS_DIR, f'run_{seed}_dir_{target_sign}')
        if not os.path.isdir(run_dir): continue
        df_ts = load_posts_timeseries(run_dir, target_sign)
        if df_ts is None: continue
        df_ts.insert(0, 'target_sign', target_sign)
        df_ts.insert(0, 'seed', seed)
        post_count_records.append(df_ts)

df_post_count = pd.concat(post_count_records, ignore_index=True)
cols = ['seed', 'target_sign', 'step'] + GROUP_ORDER
df_post_count = df_post_count[cols]
out_path = os.path.join(DEST_DIR, 'post_count_100step.csv')
df_post_count.to_csv(out_path, index=False)
print(f'post_count_100step.csv saved  ({len(df_post_count)} rows)')
df_post_count.head()

post_count_100step.csv saved  (20050 rows)


,seed,target_sign,step,far_opposite,near_opposite,neutral,near_target,far_target
0,1,1.0,0,1.60,2.25,2.69,1.98,0.91
1,1,1.0,100,1.10,2.26,2.95,1.48,0.72
2,1,1.0,200,0.97,2.45,3.19,1.55,0.96
3,1,1.0,300,0.82,2.67,3.69,1.76,0.65
4,1,1.0,400,0.90,3.07,5.45,1.68,0.62


In [5]:
# ---------------------------------------------------------------------------
# Item 3: Post probability per direction-relative class (100-step windows)
# Source: metrics/result_*.csv  columns postProbMean_0 … postProbMean_4
# NOTE: requires simulation rebuilt after Java changes (postProbMean columns added)
# ---------------------------------------------------------------------------
PP_COLS = [f'postProbMean_{i}' for i in range(5)]

def load_postprob_timeseries(run_dir, target_sign, window=WINDOW):
    metrics_dir = os.path.join(run_dir, 'metrics')
    files = glob.glob(os.path.join(metrics_dir, 'result_*.csv'))
    if not files: return None
    dfs = []
    for f in files:
        try: dfs.append(pd.read_csv(f))
        except: pass
    if not dfs: return None
    df = pd.concat(dfs).sort_values('step').reset_index(drop=True)

    missing = [c for c in PP_COLS if c not in df.columns]
    if missing:
        print(f'  [WARN] columns not found: {missing} — skipping (re-run simulation first)')
        return None

    pmap = _POS_MAP[target_sign]
    for i in range(5):
        grp = pmap[i]
        df[grp] = df.get(grp, 0.0) + df[f'postProbMean_{i}']

    # Each group might aggregate two bins (for groups that don't overlap with another bin,
    # there will be no double-counting since _POS_MAP is a bijection)
    df['window_idx'] = df['step'] // window
    agg = df.groupby('window_idx')[GROUP_ORDER].mean()
    agg['step'] = agg.index * window
    return agg.reset_index(drop=True)

post_prob_records = []
for seed in SELECTED_SEEDS:
    for target_sign in [1.0, -1.0]:
        run_dir = os.path.join(RESULTS_DIR, f'run_{seed}_dir_{target_sign}')
        if not os.path.isdir(run_dir): continue
        df_ts = load_postprob_timeseries(run_dir, target_sign)
        if df_ts is None: continue
        df_ts.insert(0, 'target_sign', target_sign)
        df_ts.insert(0, 'seed', seed)
        post_prob_records.append(df_ts)

if post_prob_records:
    df_post_prob = pd.concat(post_prob_records, ignore_index=True)
    cols = ['seed', 'target_sign', 'step'] + GROUP_ORDER
    df_post_prob = df_post_prob[cols]
    out_path = os.path.join(DEST_DIR, 'post_prob_100step.csv')
    df_post_prob.to_csv(out_path, index=False)
    print(f'post_prob_100step.csv saved  ({len(df_post_prob)} rows)')
    display(df_post_prob.head())
else:
    print('post_prob_100step.csv NOT saved — no postProbMean columns found.')
    print('Re-run simulations with the updated Java build first.')

post_prob_100step.csv saved  (20050 rows)


,seed,target_sign,step,far_opposite,near_opposite,neutral,near_target,far_target
0,1,1.0,0,0.090662,0.095105,0.096362,0.093143,0.088608
1,1,1.0,100,0.073890,0.094575,0.098120,0.082967,0.072030
2,1,1.0,200,0.066094,0.100258,0.106943,0.076865,0.062538
3,1,1.0,300,0.060346,0.113269,0.131026,0.076745,0.058081
4,1,1.0,400,0.058040,0.141699,0.177123,0.073729,0.056994


In [6]:
# ---------------------------------------------------------------------------
# Item 4: Target hub follower opinion composition (5000-step snapshots)
# Target node identified from step-20000 GEXF, then tracked back/forward.
# Follower opinion class is read from the snapshot at each respective step.
# ---------------------------------------------------------------------------
fol_records = []
print('Item 4: collecting follower compositions ...')

for seed in SELECTED_SEEDS:
    for target_sign in [1.0, -1.0]:
        run_dir = os.path.join(RESULTS_DIR, f'run_{seed}_dir_{target_sign}')
        if not os.path.isdir(run_dir): continue

        # Identify target node at step 20000
        fp_20k = _gexf_path(run_dir, 20000)
        if not fp_20k: continue
        _, tid, _ = parse_gexf_full(fp_20k)
        if tid is None: continue

        for step in GEXF_STEPS:
            fp = _gexf_path(run_dir, step)
            if not fp: continue
            G, _, agents = parse_gexf_full(fp)
            if G is None or tid not in G: continue

            followers = list(G.predecessors(tid))
            counts = {g: 0 for g in GROUP_ORDER}
            total  = 0
            for f in followers:
                oc = agents.get(f, {}).get('opinionclass', -1)
                if oc not in range(5): continue
                grp = _POS_MAP[target_sign].get(oc, 'unknown')
                if grp in counts:
                    counts[grp] += 1
                    total += 1

            row = {'seed': seed, 'target_sign': target_sign, 'step': step,
                   'total_followers': total}
            for g in GROUP_ORDER:
                row[g] = counts[g] / total if total > 0 else np.nan
            fol_records.append(row)

df_fol = pd.DataFrame(fol_records)
cols = ['seed', 'target_sign', 'step', 'total_followers'] + GROUP_ORDER
df_fol = df_fol[cols]
out_path = os.path.join(DEST_DIR, 'follower_composition_5000step.csv')
df_fol.to_csv(out_path, index=False)
print(f'follower_composition_5000step.csv saved  ({len(df_fol)} rows)')
df_fol.head()

Item 4: collecting follower compositions ...


follower_composition_5000step.csv saved  (450 rows)


,seed,target_sign,step,total_followers,far_opposite,near_opposite,neutral,near_target,far_target
0,1,1.0,0,6,0.333333,0.166667,0.166667,0.000000,0.333333
1,1,1.0,5000,63,0.000000,0.000000,0.317460,0.682540,0.000000
2,1,1.0,10000,101,0.000000,0.000000,0.316832,0.683168,0.000000
3,1,1.0,15000,138,0.000000,0.000000,0.297101,0.702899,0.000000
4,1,1.0,20000,162,0.000000,0.000000,0.327160,0.672840,0.000000


In [7]:
# ---------------------------------------------------------------------------
# Item 5: Unfollow counts by class — pre (0→20000) and post (20000→40000)
# Unfollowers classified by opinionClass at step 20000 (last pre-intervention snapshot).
# ---------------------------------------------------------------------------
unfollow_records = []
print('Item 5: collecting unfollow counts ...')

for seed in SELECTED_SEEDS:
    for target_sign in [1.0, -1.0]:
        run_dir = os.path.join(RESULTS_DIR, f'run_{seed}_dir_{target_sign}')
        if not os.path.isdir(run_dir): continue

        fp_20k = _gexf_path(run_dir, 20000)
        if not fp_20k: continue
        G_20k, tid, agents_20k = parse_gexf_full(fp_20k)
        if tid is None or G_20k is None: continue

        def _cls(nid):
            oc = agents_20k.get(nid, {}).get('opinionclass', -1)
            return _POS_MAP[target_sign].get(oc, 'unknown') if oc in range(5) else 'unknown'

        fol_20k = set(G_20k.predecessors(tid))

        fol_0 = set()
        fp_0 = _gexf_path(run_dir, 0)
        if fp_0:
            G_0, _, _ = parse_gexf_full(fp_0)
            if G_0 is not None and tid in G_0:
                fol_0 = set(G_0.predecessors(tid))

        fol_40k = set()
        fp_40k = _gexf_path(run_dir, 40000)
        if fp_40k:
            G_40k, _, _ = parse_gexf_full(fp_40k)
            if G_40k is not None and tid in G_40k:
                fol_40k = set(G_40k.predecessors(tid))

        for window, unfol in [('pre_0_20000', fol_0 - fol_20k),
                               ('post_20000_40000', fol_20k - fol_40k)]:
            counts = {g: 0 for g in GROUP_ORDER}
            for nid in unfol:
                grp = _cls(nid)
                if grp in counts:
                    counts[grp] += 1
            row = {'seed': seed, 'target_sign': target_sign, 'window': window}
            row.update(counts)
            unfollow_records.append(row)

df_unfollow = pd.DataFrame(unfollow_records)
cols = ['seed', 'target_sign', 'window'] + GROUP_ORDER
df_unfollow = df_unfollow[cols]
out_path = os.path.join(DEST_DIR, 'unfollow_by_class.csv')
df_unfollow.to_csv(out_path, index=False)
print(f'unfollow_by_class.csv saved  ({len(df_unfollow)} rows)')
df_unfollow.head()

Item 5: collecting unfollow counts ...


unfollow_by_class.csv saved  (100 rows)


,seed,target_sign,window,far_opposite,near_opposite,neutral,near_target,far_target
0,1,1.0,pre_0_20000,2,1,1,0,2
1,1,1.0,post_20000_40000,0,0,53,109,0
2,1,-1.0,pre_0_20000,2,0,1,1,2
3,1,-1.0,post_20000_40000,0,109,53,0,0
4,3,1.0,pre_0_20000,0,0,1,0,0


In [8]:
# ---------------------------------------------------------------------------
# Item 6: Agents with increased posting probability by class
# pre  window: steps 15000 & 20000 (snapshots just before/at manipulation)
# post window: steps 35000 & 40000 (snapshots near end of simulation)
# opinionClass assigned from step 15000 (last pre-manipulation class snapshot).
# Target node identified from step 20000.
# ---------------------------------------------------------------------------
_A6_PRE  = [15000, 20000]
_A6_POST = [35000, 40000]

def _load_agents(run_dir, step):
    fp = _gexf_path(run_dir, step)
    if not fp: return {}
    _, _, agents = parse_gexf_full(fp)
    return agents

pp_increase_records = []
print('Item 6: counting agents with increased postProb ...')

for seed in SELECTED_SEEDS:
    for target_sign in [1.0, -1.0]:
        run_dir = os.path.join(RESULTS_DIR, f'run_{seed}_dir_{target_sign}')
        if not os.path.isdir(run_dir): continue

        # Identify target from step 20000
        fp_20k = _gexf_path(run_dir, 20000)
        if not fp_20k: continue
        _, tid, _ = parse_gexf_full(fp_20k)
        if tid is None: continue

        # opinionClass reference: step 15000 (pre-manipulation)
        agents_ref = _load_agents(run_dir, 15000)
        if not agents_ref: continue

        pre_snaps  = [_load_agents(run_dir, s) for s in _A6_PRE]
        post_snaps = [_load_agents(run_dir, s) for s in _A6_POST]
        pre_snaps  = [d for d in pre_snaps  if d]
        post_snaps = [d for d in post_snaps if d]
        if not pre_snaps or not post_snaps: continue

        # per-class counters
        n_increased = {g: 0 for g in GROUP_ORDER}
        n_total     = {g: 0 for g in GROUP_ORDER}

        for nid, ref in agents_ref.items():
            if nid == tid: continue  # skip target
            oc = ref.get('opinionclass', -1)
            if oc not in range(5): continue
            grp = _POS_MAP[target_sign].get(oc, 'unknown')
            if grp not in GROUP_ORDER: continue

            pre_pp  = [sn[nid]['postprob'] for sn in pre_snaps
                       if nid in sn and not np.isnan(sn[nid]['postprob'])]
            post_pp = [sn[nid]['postprob'] for sn in post_snaps
                       if nid in sn and not np.isnan(sn[nid]['postprob'])]
            if not pre_pp or not post_pp: continue

            n_total[grp] += 1
            if np.mean(post_pp) > np.mean(pre_pp):
                n_increased[grp] += 1

        for grp in GROUP_ORDER:
            total = n_total[grp]
            pp_increase_records.append({
                'seed': seed, 'target_sign': target_sign,
                'relative_group': grp,
                'n_increased': n_increased[grp],
                'n_total':     total,
                'rate':        n_increased[grp] / total if total > 0 else np.nan,
            })

df_pp_inc = pd.DataFrame(pp_increase_records)
out_path = os.path.join(DEST_DIR, 'posting_prob_increase_by_class.csv')
df_pp_inc.to_csv(out_path, index=False)
print(f'posting_prob_increase_by_class.csv saved  ({len(df_pp_inc)} rows)')
df_pp_inc.head(10)

Item 6: counting agents with increased postProb ...


posting_prob_increase_by_class.csv saved  (250 rows)


,seed,target_sign,relative_group,n_increased,n_total,rate
0,1,1.0,far_opposite,45,150,0.300000
1,1,1.0,near_opposite,13,235,0.055319
2,1,1.0,neutral,9,287,0.031359
3,1,1.0,near_target,72,201,0.358209
4,1,1.0,far_target,34,126,0.269841
5,1,-1.0,far_opposite,33,126,0.261905
6,1,-1.0,near_opposite,78,201,0.388060
7,1,-1.0,neutral,9,287,0.031359
8,1,-1.0,near_target,15,235,0.063830
9,1,-1.0,far_target,48,150,0.320000


In [9]:
# ---------------------------------------------------------------------------
# Item 7: Target hub structural & opinion metrics (5000-step snapshots)
# Metrics mirror structural-backfire-analysis.ipynb:
#   in_degree, betweenness (normalized), clustering, pagerank, eigenvector
#   target_opinion, follower_mean_opinion, follower_opinion_std,
#   follower_homophily_frac, follower_dir_alignment
# Target node identified from step 20000, then tracked across all snapshots.
# ---------------------------------------------------------------------------

def _parse_gexf_with_opinion(fpath):
    """Parse GEXF, returning (DiGraph, target_id, {nid: {opinionclass, opinion}})."""
    try:
        tree = ET.parse(fpath); root = tree.getroot(); _strip_ns(root)
        attr_map = {a.get('id'): a.get('title')
                    for a in root.findall(".//attributes[@class='node']/attribute")}
        G = nx.DiGraph()
        target_id = None
        agents = {}
        for node in root.findall('.//node'):
            nid = node.get('id')
            G.add_node(nid)
            d = {'opinionclass': -1, 'opinion': np.nan}
            for av in node.findall('.//attvalue'):
                t, v = attr_map.get(av.get('for'), av.get('for')), av.get('value', '')
                if t == 'opinionClass':
                    try: d['opinionclass'] = int(v)
                    except: pass
                elif t == 'opinion':
                    try: d['opinion'] = float(v)
                    except: pass
                elif t == 'target' and v.lower() == 'true':
                    target_id = nid
            agents[nid] = d
        for edge in root.findall('.//edge'):
            G.add_edge(edge.get('source'), edge.get('target'))
        return G, target_id, agents
    except Exception:
        return None, None, {}


def _target_structural_metrics(G, tid):
    """Node-level structural indices for the target node."""
    result = {'in_degree': np.nan, 'betweenness': np.nan,
              'clustering': np.nan, 'pagerank': np.nan, 'eigenvector': np.nan}
    if G is None or tid not in G:
        return result
    result['in_degree']   = G.in_degree(tid)
    result['betweenness'] = nx.betweenness_centrality(G, normalized=True).get(tid, np.nan)
    result['clustering']  = nx.clustering(G, tid)
    result['pagerank']    = nx.pagerank(G, alpha=0.85, max_iter=200).get(tid, np.nan)
    try:
        result['eigenvector'] = nx.eigenvector_centrality(G, max_iter=1000, tol=1e-6).get(tid, np.nan)
    except nx.PowerIterationFailedConvergence:
        result['eigenvector'] = np.nan
    return result


def _target_follower_opinion_metrics(G, tid, agents, target_sign):
    """Follower opinion homophily metrics for the target node."""
    result = {'target_opinion': np.nan, 'follower_mean_opinion': np.nan,
              'follower_opinion_std': np.nan, 'follower_homophily_frac': np.nan,
              'follower_dir_alignment': np.nan}
    if G is None or tid not in G:
        return result
    target_op = agents.get(tid, {}).get('opinion', np.nan)
    result['target_opinion'] = target_op
    fol_ops = np.array([agents[f]['opinion'] for f in G.predecessors(tid)
                        if f in agents and not np.isnan(agents[f]['opinion'])])
    if len(fol_ops) == 0:
        return result
    result['follower_mean_opinion'] = float(np.mean(fol_ops))
    result['follower_opinion_std']  = float(np.std(fol_ops))
    if not np.isnan(target_op):
        result['follower_homophily_frac'] = float(np.mean(np.sign(fol_ops) == np.sign(target_op)))
    result['follower_dir_alignment'] = float(np.mean(np.sign(fol_ops) == np.sign(target_sign)))
    return result


hub_records = []
print('Item 7: collecting target hub metrics ...')

for seed in SELECTED_SEEDS:
    for target_sign in [1.0, -1.0]:
        run_dir = os.path.join(RESULTS_DIR, f'run_{seed}_dir_{target_sign}')
        if not os.path.isdir(run_dir):
            continue

        fp_20k = _gexf_path(run_dir, 20000)
        if not fp_20k:
            continue
        _, tid, _ = parse_gexf_full(fp_20k)
        if tid is None:
            continue

        for step in GEXF_STEPS:
            fp = _gexf_path(run_dir, step)
            if not fp:
                continue
            G, _, agents = _parse_gexf_with_opinion(fp)
            if G is None or tid not in G:
                continue

            struct    = _target_structural_metrics(G, tid)
            homophily = _target_follower_opinion_metrics(G, tid, agents, target_sign)
            hub_records.append({
                'seed': seed, 'target_sign': target_sign, 'step': step,
                **struct, **homophily,
            })

df_hub = pd.DataFrame(hub_records)
out_path = os.path.join(DEST_DIR, 'target_hub_metrics_5000step.csv')
df_hub.to_csv(out_path, index=False)
print(f'target_hub_metrics_5000step.csv saved  ({len(df_hub)} rows)')
df_hub.head()


Item 7: collecting target hub metrics ...


target_hub_metrics_5000step.csv saved  (450 rows)


,seed,target_sign,step,in_degree,betweenness,clustering,pagerank,eigenvector,target_opinion,follower_mean_opinion,follower_opinion_std,follower_homophily_frac,follower_dir_alignment
0,1,1.0,0,6,0.004257,0.000000,0.001985,5.303329e-02,0.152121,-0.089813,0.643898,0.333333,0.333333
1,1,1.0,5000,63,0.000731,0.062215,0.002218,1.631648e-03,0.135058,0.268924,0.116798,1.000000,1.000000
2,1,1.0,10000,101,0.000353,0.052088,0.003785,9.880653e-06,0.139031,0.269302,0.110942,1.000000,1.000000
3,1,1.0,15000,138,0.000486,0.048819,0.005659,6.566082e-06,0.138515,0.270986,0.106719,1.000000,1.000000
4,1,1.0,20000,162,0.000784,0.052673,0.007823,2.274359e-66,0.140075,0.261939,0.107127,1.000000,1.000000


In [10]:
# ---------------------------------------------------------------------------
# Item 8: Network connectivity metrics (5000-step snapshots)
#
# All metrics computed on the undirected projection of the directed follow-graph.
# Target node identified from step 20000, tracked across all snapshots.
#
# Columns
# -------
# lambda2_G              -- algebraic connectivity (Fiedler lambda2) of full graph
# lambda2_G_minus_target -- lambda2 of graph with target hub removed
# delta_lambda2          -- lambda2_G - lambda2_G_minus_target
#                           (hub marginal contribution to global connectivity;
#                            large value = hub is load-bearing bridge)
# target_is_ap           -- 1 if target hub is an articulation point, else 0
#
# Implementation notes
# --------------------
# * lambda2 is computed via nx.algebraic_connectivity (ARPACK sparse eigensolver)
#   on the undirected projection. Bridge is an undirected concept: we ask
#   whether removing the hub disconnects any part of the graph regardless of
#   follow-edge direction.
# * Self-loops are stripped before computation (they do not affect lambda2 but
#   may cause degenerate Laplacian rows).
# * Disconnected graph => lambda2 = 0.0 (by definition). Graphs with < 2 nodes
#   after hub removal => lambda2 = nan.
# * Articulation point check runs in O(V + E) via DFS (Tarjan 1972).
# ---------------------------------------------------------------------------

def _algebraic_connectivity(G_und):
    """
    Fiedler value (lambda2) of the undirected graph Laplacian.
    Returns 0.0 for disconnected graphs, nan for graphs with < 2 nodes.
    """
    n = G_und.number_of_nodes()
    if n < 2:
        return np.nan
    if not nx.is_connected(G_und):
        return 0.0
    try:
        return nx.algebraic_connectivity(G_und, tol=1e-6, seed=42)
    except Exception:
        return np.nan


conn_records = []
print("Item 8: collecting connectivity metrics ...")

for seed in SELECTED_SEEDS:
    for target_sign in [1.0, -1.0]:
        run_dir = os.path.join(RESULTS_DIR, f"run_{seed}_dir_{target_sign}")
        if not os.path.isdir(run_dir):
            continue

        # Identify target node from step 20000
        fp_20k = _gexf_path(run_dir, 20000)
        if not fp_20k:
            continue
        _, tid, _ = parse_gexf_full(fp_20k)
        if tid is None:
            continue

        for step in GEXF_STEPS:
            fp = _gexf_path(run_dir, step)
            if not fp:
                continue
            G, _, _ = parse_gexf_full(fp)
            if G is None or tid not in G:
                continue

            # Undirected projection; strip self-loops
            G_und = G.to_undirected()
            G_und.remove_edges_from(nx.selfloop_edges(G_und))

            # --- lambda2(G) ---
            l2_G = _algebraic_connectivity(G_und)

            # --- Articulation point (O(V+E) DFS) ---
            ap_set = set(nx.articulation_points(G_und))
            is_ap  = int(tid in ap_set)

            # --- lambda2(G \ {target hub}) and delta_lambda2 ---
            G_und_mt = G_und.copy()
            G_und_mt.remove_node(tid)
            l2_mt = _algebraic_connectivity(G_und_mt)

            delta = (l2_G - l2_mt) if not (np.isnan(l2_G) or np.isnan(l2_mt)) else np.nan

            conn_records.append({
                "seed":                   seed,
                "target_sign":            target_sign,
                "step":                   step,
                "lambda2_G":              l2_G,
                "lambda2_G_minus_target": l2_mt,
                "delta_lambda2":          delta,
                "target_is_ap":           is_ap,
            })

df_conn = pd.DataFrame(conn_records)
out_path = os.path.join(DEST_DIR, "connectivity_5000step.csv")
df_conn.to_csv(out_path, index=False)
print(f"connectivity_5000step.csv saved  ({len(df_conn)} rows)")
df_conn.head()

Item 8: collecting connectivity metrics ...


connectivity_5000step.csv saved  (450 rows)


,seed,target_sign,step,lambda2_G,lambda2_G_minus_target,delta_lambda2,target_is_ap
0,1,1.0,0,0.0,0.0,0.0,0
1,1,1.0,5000,0.0,0.0,0.0,0
2,1,1.0,10000,0.0,0.0,0.0,0
3,1,1.0,15000,0.0,0.0,0.0,0
4,1,1.0,20000,0.0,0.0,0.0,0


In [11]:
# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
saved = [
    'selected_seeds.json',
    'nw_stats_5000step.csv',
    'post_count_100step.csv',
    'post_prob_100step.csv',
    'follower_composition_5000step.csv',
    'unfollow_by_class.csv',
    'posting_prob_increase_by_class.csv',
    'target_hub_metrics_5000step.csv',
    'connectivity_5000step.csv',
]
print(f'\nAll outputs in: {DEST_DIR}\n')
for fname in saved:
    fpath = os.path.join(DEST_DIR, fname)
    status = '✓' if os.path.exists(fpath) else '✗ MISSING'
    size = f'{os.path.getsize(fpath)/1024:.1f} KB' if os.path.exists(fpath) else ''
    print(f'  {status}  {fname}  {size}')


All outputs in: ./results/summary/ER/p_0.003/dad3c91b

  ✓  selected_seeds.json  0.2 KB
  ✓  nw_stats_5000step.csv  39.8 KB
  ✓  post_count_100step.csv  760.7 KB
  ✓  post_prob_100step.csv  1567.8 KB
  ✓  follower_composition_5000step.csv  20.2 KB
  ✓  unfollow_by_class.csv  3.3 KB
  ✓  posting_prob_increase_by_class.csv  11.0 KB
  ✓  target_hub_metrics_5000step.csv  56.8 KB
  ✓  connectivity_5000step.csv  12.0 KB
